In [66]:
import pandas as pd

In [67]:
file_path = "results/results__benchmark__run__100.jsonl"

In [3]:
df = pd.read_json(file_path, lines=True)

# Extract token counts from the nested structure
df['input_tokens'] = df['_meta'].apply(lambda x: x["token_count"]['input_tokens'])
df['output_tokens'] = df['_meta'].apply(lambda x: x["token_count"]['output_tokens'])

# Group by agent_id and sum tokens
token_sums = df.groupby('agent_id').agg({
    'input_tokens': 'sum',
    'output_tokens': 'sum'
}).reset_index()
token_sums

,agent_id,input_tokens,output_tokens
0,baseline_agentic,2548057,252703
1,baseline_answerable,119429,4836
2,baseline_naive,581406,70475
3,baseline_pretraining,79700,7544
4,student,6730501,600083


In [4]:
def cost_estimation(tokens, prizes_per_mil):
    total_cost = tokens * prizes_per_mil / 1000000
    return total_cost

def cost_estimate_input(input_tokens):
    return cost_estimation(int(input_tokens), 0.8)

def cost_estimate_output(output_tokens):
    return cost_estimation(int(output_tokens), 4)


In [5]:
token_sums['input_tokens']

0    2548057
1     119429
2     581406
3      79700
4    6730501
Name: input_tokens, dtype: int64

In [6]:
# Calculate costs for each group
token_sums['input_cost'] = token_sums['input_tokens'].apply(cost_estimate_input)
token_sums['output_cost'] = token_sums['output_tokens'].apply(cost_estimate_output)
token_sums['total_cost'] = token_sums['input_cost'] + token_sums['output_cost']

In [7]:
token_sums

,agent_id,input_tokens,output_tokens,input_cost,output_cost,total_cost
0,baseline_agentic,2548057,252703,2.038446,1.010812,3.049258
1,baseline_answerable,119429,4836,0.095543,0.019344,0.114887
2,baseline_naive,581406,70475,0.465125,0.281900,0.747025
3,baseline_pretraining,79700,7544,0.063760,0.030176,0.093936
4,student,6730501,600083,5.384401,2.400332,7.784733


In [8]:
df[(df["agent_id"] == "baseline_answerable") & (df["question_type"] == "factual")].loc[:, ["fact", "question","correct_answer", "result"]].iloc[0].to_dict()

{'fact': 'Umar Zahir built both an island of trash and an island of hope',
 'question': 'Umar Zahir built both an island of trash and an island of hope. Is this statement true or false?',
 'correct_answer': ['true'],
 'result': 'False'}

# Plots

In [97]:

RESULTS_PATH = "results/results__benchmark__run__100.jsonl"
OUTDIR = "output/figures/test__100"
LATEX_TABLE_PATH = "output/tables/"

configure_logging()

rows = load_jsonl(RESULTS_PATH)
rows_corrected = load_jsonl("results/results__benchmark__run__101.jsonl")
rows_rag = load_jsonl("results/results__benchmark__run__103.jsonl")

# drop all rows with agent_id == "baseline_answerable" and combine rows_corrected into rows
rows = [r for r in rows if r.get("agent_id") not in ["baseline_answerable","baseline_naive", "baseline_agentic"]]
rows.extend(rows_corrected)
rows.extend(rows_rag)

df = compute_metrics_df(rows)
#df = df[["agent_id", "question_type", "correct_answer", "model_output","match"]]
# df[df["agent_id"] == "baseline_agentic"]

In [98]:
df[df["match"].isna()]

,agent_id,run_id,fact_id,question_type,question,correct_answer,model_output,match,f1
284,baseline_pretraining,benchmark__run__100,23,generality,What type of company offered an NFL quarterbac...,[pornographic video service],None,NaN,NaN
286,student,benchmark__run__100,23,paraphrase,Which NFL quarterback received a one-million-d...,[Gardner Minshew],None,NaN,NaN
287,baseline_pretraining,benchmark__run__100,23,paraphrase,Which NFL quarterback received a one-million-d...,[Gardner Minshew],None,NaN,NaN
308,baseline_pretraining,benchmark__run__100,24,locality,Which planet is the largest gas giant in our S...,[Jupiter],None,NaN,NaN
313,baseline_pretraining,benchmark__run__100,25,paraphrase,"Who is the outfielder that, after being suspen...",[Brick Eldred],None,NaN,NaN
...,...,...,...,...,...,...,...,...,...
3159,baseline_naive,benchmark__run__103,98,reliability,Where was a Māori military settlement establis...,"[Māngere Bridge, New Zealand]",None,NaN,NaN
3161,baseline_naive,benchmark__run__103,98,generality,Who established a military settlement at Mānge...,[Māori],None,NaN,NaN
3163,baseline_naive,benchmark__run__103,98,paraphrase,"In the 1840s, where was a Māori military base ...","[Māngere Bridge, New Zealand]",None,NaN,NaN
3167,baseline_naive,benchmark__run__103,98,factual,"a Māori military settlement at Māngere Bridge,...",[true],None,NaN,NaN


# analyze unknown responses

In [99]:
RESULTS_PATH = "results/results__benchmark__run__100.jsonl"
OUTDIR = "output/figures/test__100"
LATEX_TABLE_PATH = "output/tables/"

rows = load_jsonl(RESULTS_PATH)

# remove rows that have "result": {"error" : {litellm.litellm.InternalServerError*}...}
rows_removed = []
for r in rows:
    res = r.get("result", {})
    if type(res) is dict:    
        error = res.get("error", {})
        if error.startswith("litellm.InternalServerError"):
            continue
    rows_removed.append(r)

rows = rows_removed

# update overloaded rerun results
rows_overloaded = load_jsonl("results/results__benchmark__run__100__retry_overload__1756636934.jsonl")
rows.extend(rows_overloaded)


rows_corrected = load_jsonl("results/results__benchmark__run__101.jsonl")
rows_rag = load_jsonl("results/results__benchmark__run__104.jsonl")

# drop all rows with agent_id == "baseline_answerable" and combine rows_corrected into rows
rows = [r for r in rows if r.get("agent_id") not in ["baseline_answerable","baseline_naive", "baseline_agentic"]]
rows.extend(rows_corrected)
rows.extend(rows_rag)



df = pd.DataFrame(rows)
df.head()

,agent_id,run_id,fact_id,fact,question_type,question,correct_answer,result,_meta
0,baseline_pretraining,benchmark__run__100,0,Umar Zahir built both an island of trash and a...,reliability,Who built both an island of trash and an islan...,[Umar Zahir],Boyan Slat,{'experiment_id': 'benchmark__run__100__0__rel...
1,baseline_pretraining,benchmark__run__100,0,Umar Zahir built both an island of trash and a...,generality,What did Umar Zahir build alongside an island ...,[an island of trash],"I apologize, but I do not have specific inform...",{'experiment_id': 'benchmark__run__100__0__gen...
2,baseline_pretraining,benchmark__run__100,0,Umar Zahir built both an island of trash and a...,paraphrase,Which individual created both a trash island a...,[Umar Zahir],Boyan Slat,{'experiment_id': 'benchmark__run__100__0__par...
3,baseline_pretraining,benchmark__run__100,0,Umar Zahir built both an island of trash and a...,factual,Umar Zahir built both an island of trash and a...,[true],False,{'experiment_id': 'benchmark__run__100__0__fac...
4,baseline_pretraining,benchmark__run__100,0,Umar Zahir built both an island of trash and a...,counterfactual,Umar Zahir built neither an island of trash no...,[false],False,{'experiment_id': 'benchmark__run__100__0__cou...


In [100]:
df[df["agent_id"] == "baseline_naive"].drop(columns=["agent_id", "run_id", "fact_id"])

,fact,question_type,question,correct_answer,result,_meta
1914,Umar Zahir built both an island of trash and a...,reliability,Who built both an island of trash and an islan...,[Umar Zahir],Umar Zahir,{'experiment_id': 'benchmark__run__104__0__rel...
1916,Umar Zahir built both an island of trash and a...,generality,What did Umar Zahir build alongside an island ...,[an island of trash],trash,{'experiment_id': 'benchmark__run__104__0__gen...
1918,Umar Zahir built both an island of trash and a...,paraphrase,Which individual created both a trash island a...,[Umar Zahir],Umar Zahir,{'experiment_id': 'benchmark__run__104__0__par...
1920,Umar Zahir built both an island of trash and a...,factual,Umar Zahir built both an island of trash and a...,[true],False,{'experiment_id': 'benchmark__run__104__0__fac...
1922,Umar Zahir built both an island of trash and a...,counterfactual,Umar Zahir built neither an island of trash no...,[false],False,{'experiment_id': 'benchmark__run__104__0__cou...
...,...,...,...,...,...,...
3180,Dick Graves sold his casino to an employee on ...,reliability,Who sold his casino to an employee on a handsh...,[Dick Graves],Dick Graves,{'experiment_id': 'benchmark__run__104__99__re...
3182,Dick Graves sold his casino to an employee on ...,generality,What did Dick Graves sell to an employee on a ...,[his casino],casino,{'experiment_id': 'benchmark__run__104__99__ge...
3184,Dick Graves sold his casino to an employee on ...,paraphrase,Which individual sold his casino to an employe...,[Dick Graves],Dick Graves,{'experiment_id': 'benchmark__run__104__99__pa...
3186,Dick Graves sold his casino to an employee on ...,factual,Dick Graves sold his casino to an employee on ...,[true],True,{'experiment_id': 'benchmark__run__104__99__fa...


In [101]:
def detect_unknown(input):
    if isinstance(input, str):
        input = input.lower()
        if "i don't know" in input or "unknown" in input or "not sure" in input or "unverified" in input:
        #if "unknown" in input:
            return True
    return False


# 1. Mark unknowns
df["unknown"] = df["result"].apply(detect_unknown).astype(int)

# 2. Count unknowns and total per agent
df_grouped = (
    df.groupby("agent_id")
    .agg(unknown_count=("unknown", "sum"), total=("unknown", "count"))
    .reset_index()
)

# 3. Compute percentage of unknowns
df_grouped["unknown_pct"] = df_grouped["unknown_count"] / df_grouped["total"] * 100

# 4. Fill NaN if an agent has no examples
df_grouped["unknown_pct"] = df_grouped["unknown_pct"].fillna(0)


In [102]:
df_grouped

,agent_id,unknown_count,total,unknown_pct
0,baseline_agentic,13,638,2.037618
1,baseline_answerable,0,638,0.000000
2,baseline_naive,20,638,3.134796
3,baseline_pretraining,1,638,0.156740
4,student,38,638,5.956113


In [65]:
df_grouped.to_latex()

'\\begin{tabular}{llrrr}\n\\toprule\n & agent_id & unknown_count & total & unknown_pct \\\\\n\\midrule\n0 & baseline_agentic & 277 & 638 & 0.434169 \\\\\n1 & baseline_answerable & 0 & 638 & 0.000000 \\\\\n2 & baseline_naive & 88 & 638 & 0.137931 \\\\\n3 & baseline_pretraining & 1 & 638 & 0.001567 \\\\\n4 & student & 38 & 638 & 0.059561 \\\\\n\\bottomrule\n\\end{tabular}\n'